# The Silent Heist
Reproduce the fictional bank simulation and analyze confirmed fraud. Run from repository root or python folder.

In [ ]:
from pathlib import Path
import sys, sqlite3
import pandas as pd
import matplotlib.pyplot as plt
root=Path.cwd() if (Path.cwd()/'data').exists() else Path.cwd().parent
sys.path.insert(0,str(root/'python'))
from generate import generate
customers, transactions = generate(root)
assert len(transactions)==20000 and transactions.transaction_id.is_unique
assert transactions.confirmed_fraud.sum()==160
assert (transactions.net_loss_cad>=0).all()

In [ ]:
df=transactions.merge(customers,on='customer_id',validate='many_to_one')
df['month']=df.date.str[:7]
print(df[['amount_cad','gross_loss_cad','recovered_cad','net_loss_cad']].sum())
print('Fraud rate:',df.confirmed_fraud.mean())
print('Alert recall:',df.loc[df.confirmed_fraud==1,'alert_flag'].mean())
print('Alert precision:',df.loc[df.alert_flag==1,'confirmed_fraud'].mean())

In [ ]:
df.groupby('month').gross_loss_cad.sum().plot(kind='line', marker='o', title='Monthly gross fraud loss (CAD)'); plt.show()
df.groupby('province').net_loss_cad.sum().sort_values().plot.barh(title='Net loss by province (CAD)'); plt.show()

In [ ]:
con=sqlite3.connect(':memory:')
transactions.to_sql('transactions',con,index=False)
customers.to_sql('customers',con,index=False)
for query in (root/'sql/fraud_analysis.sql').read_text().split(';'):
    if query.strip(): display(pd.read_sql_query(query,con))

## Caveat
Fraud growth is deliberately seeded, and alert performance is simulated, not a trained or validated production model.